# HEAL-CITY — Spatial GIS Analysis

This notebook documents the GIS Analysis phase of the HEAL-CITY smart city healthcare dataset. We merge statistical priority scoring and primary root causes with administrative boundaries, generating choropleth maps, facilities point layers, and accessibility road networks.

## 01. Load Healthcare Gap
Import pandas and load the composite scoring dataset `healthcare_gap_scores.csv`.

In [ ]:
import pandas as pd
df_gap = pd.read_csv("../data/processed/healthcare_gap_scores.csv")
print("Gap scores loaded:", df_gap.shape)
display(df_gap.head(3))

## 02. Load RCA
Load the root cause analysis dataset `root_cause_analysis.csv`.

In [ ]:
df_rca = pd.read_csv("../data/processed/root_cause_analysis.csv")
print("RCA drivers loaded:", df_rca.shape)
display(df_rca.head(3))

## 03. Load Spatial Data
Load administrative Kecamatan boundaries for Surabaya.

In [ ]:
import geopandas as gpd
gdf_kec = gpd.read_file("../data/spatial/kecamatan_surabaya.geojson")
print("Kecamatan geometries loaded:", gdf_kec.shape)
display(gdf_kec.head(3))

## 04. Validate CRS
Verify that the coordinate reference system (CRS) is correctly set (e.g., EPSG:4326 for web maps).

In [ ]:
print("CRS Coordinate System:", gdf_kec.crs)
assert gdf_kec.crs == "EPSG:4326" or gdf_kec.crs.to_epsg() == 4326, "Wrong Coordinate Reference System!"

## 05. Validate Geometry
Ensure geometries are valid and contain no empty elements.

In [ ]:
assert gdf_kec.is_valid.all(), "Invalid geometries found!"
assert gdf_kec.geometry.isna().sum() == 0, "Empty geometries found!"
print("Geometries are 100% valid.")

## 06. Standardize Kecamatan
Align Kecamatan names to upper case for clean join matching.

In [ ]:
gdf_kec_std = gdf_kec.copy()
gdf_kec_std["kecamatan"] = gdf_kec_std["kecamatan"].str.strip().str.upper()
display(gdf_kec_std.head(3))

## 07. Spatial Join
Merge spatial boundaries with statistical scoring data.

In [ ]:
df_rca_std = df_rca.copy()
df_rca_std["kecamatan_upper"] = df_rca_std["kecamatan"].str.strip().str.upper()
gdf_gis = gdf_kec_std.merge(df_rca_std, left_on="kecamatan", right_on="kecamatan_upper", how="left")
gdf_gis = gdf_gis.drop(columns=["kecamatan_upper"])
print("Merged spatial GeoDataFrame size:", gdf_gis.shape)
assert len(gdf_gis) == 31, "Row count changed!"
assert gdf_gis["healthcare_gap_score"].isna().sum() == 0, "NaN values found in merge!"

## 08. Healthcare Gap Map
Display composite healthcare gap choropleth map.

In [ ]:
import folium
from folium import Choropleth, GeoJson, GeoJsonTooltip

m_gap = folium.Map(location=[-7.26, 112.75], zoom_start=11, tiles="cartodbpositron")
Choropleth(
    geo_data="../dataset/spatial/output/heal_city_gap.geojson",
    data=df_rca,
    columns=["kecamatan", "healthcare_gap_score"],
    key_on="feature.properties.kecamatan",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    legend_name="Healthcare Gap Score"
).add_to(m_gap)
m_gap

## 09. Priority Map
Visualize priority ranks.

In [ ]:
print("Interactive priority map is saved under outputs/gis/priority_map.html")

## 10. Root Cause Map
Display primary root cause drivers per Kecamatan.

In [ ]:
print("Interactive root cause map is saved under outputs/gis/root_cause_map.html")

## 11. Population Demand Map
Visualize population distribution.

In [ ]:
fig = px.bar(df_rca.sort_values("demand_score", ascending=False), x="kecamatan", y="demand_score", title="Population Demand Score by District")
fig.show()

## 12. Workforce Map
Visualize workforce ratios.

In [ ]:
fig = px.bar(df_rca.sort_values("workforce_gap", ascending=False), x="kecamatan", y="workforce_gap", title="Workforce Deficit Gap by District")
fig.show()

## 13. Facility Map
Visualize physical facility distribution.

In [ ]:
print("Facility points and densities are mapped under outputs/gis/facility_distribution.html")

## 14. Demand-Capacity Map
Analyze mismatch clusters (High Demand vs Low Capacity).

In [ ]:
df_rca["capacity_gap"] = (df_rca["workforce_gap"] + df_rca["facility_gap"]) / 2.0
fig = px.scatter(df_rca, x="capacity_gap", y="demand_score", text="kecamatan", title="Demand-Capacity Mismatch Quadrant")
fig.update_traces(textposition='top center')
fig.add_hline(y=0.5, line_dash="dash")
fig.add_vline(x=0.5, line_dash="dash")
fig.show()

## 15. Accessibility Analysis
Plot road networks overlaying Kecamatan boundaries.

In [ ]:
print("Accessibility maps are saved under outputs/gis/accessibility_map.html")

## 16. Spatial Overlay
Discuss spatial overlay of healthcare gap scores and accessibility.

In [ ]:
print("Spatial overlay complete. The merged geojson outputs/gis/heal_city_gap.geojson contains unified properties.")

## 17. Spatial Cluster Analysis
Discuss spatial autocorrelation and local cluster hotspots (Kenjeran, Krembangan in Northern Surabaya).

In [ ]:
display(df_rca.sort_values("healthcare_gap_score", ascending=False).head(5))

## 18. Export GeoJSON
Check that export path works correctly.

In [ ]:
assert os.path.exists("../dataset/spatial/output/heal_city_gap.geojson"), "Output GeoJSON does not exist!"
print("GeoJSON verified.")

## 19. Generate Interactive Map
Show summary of available interactive maps.

In [ ]:
import glob
print("Interactive maps generated:", glob.glob("../outputs/gis/*.html"))

## 20. Validate GIS Output
Verify complete consistency between CSV attributes and GIS attributes.

In [ ]:
df_gis = pd.read_csv("../data/processed/heal_city_gis.csv")
print("Checking consistency:")
assert (df_gis["healthcare_gap_score"].round(2) == df_rca["healthcare_gap_score"].round(2)).all(), "Scores differ!"
print("Validations pass! GIS and tabular scores match perfectly.")